# DOT Budget vs. Spending Analysis (FY2017–FY2027)

Compares adopted budget, modified budget, and actual spending for the
NYC Department of Transportation, one step at a time. Each step prints
its own output for verification before moving on. **No modeling yet —
stops after Step 5.**

In [1]:
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

DATA_DIR = "raw datasets"
BUDGET_PATH = f"{DATA_DIR}/DOT_Budget_2017_2027.csv"
SPENDING_PATH = f"{DATA_DIR}/DOT_Spending_Merged.csv"


## Step 1: Load & Inspect

No assumptions about column names yet — just load and look.

In [2]:
budget_raw = pd.read_csv(BUDGET_PATH)

print("=== DOT_Budget_2017_2027.csv ===")
print("Shape:", budget_raw.shape)
print("\nColumns:", list(budget_raw.columns))
print("\nDtypes:")
print(budget_raw.dtypes)
print("\nHead:")
budget_raw.head()


=== DOT_Budget_2017_2027.csv ===
Shape: (334806, 9)

Columns: ['Adopted', 'Agency', 'Budget Code', 'Expense Category', 'Modified', 'Post Adjustments', 'Pre-Encumbered', 'Year', 'Source_File']

Dtypes:
Adopted             float64
Agency                  str
Budget Code             str
Expense Category        str
Modified            float64
Post Adjustments    float64
Pre-Encumbered      float64
Year                  int64
Source_File             str
dtype: object

Head:


,Adopted,Agency,Budget Code,Expense Category,Modified,Post Adjustments,Pre-Encumbered,Year,Source_File
0,56261392.0,Department of Transportation,4125,HEAT LIGHT & POWER,53742028.0,0.00,0.0,2017,nyc-data-feed DOT 2017.csv
1,39071937.0,Department of Transportation,2002,SUPPLIES + MATERIALS - GENERAL,42563150.0,120.94,0.0,2017,nyc-data-feed DOT 2017.csv
2,32955700.0,Department of Transportation,4122,MAINT & OPER OF INFRASTRUCTURE,30104225.0,0.00,0.0,2017,nyc-data-feed DOT 2017.csv
3,27844600.0,Department of Transportation,3100,FULL YEAR POSITIONS,29090761.0,0.00,0.0,2017,nyc-data-feed DOT 2017.csv
4,30256607.0,Department of Transportation,1270,RENTALS - LAND BLDGS & STRUCTS,25870624.0,0.00,0.0,2017,nyc-data-feed DOT 2017.csv


In [3]:
spending_raw = pd.read_csv(SPENDING_PATH)

print("=== DOT_Spending_Merged.csv ===")
print("Shape:", spending_raw.shape)
print("\nColumns:", list(spending_raw.columns))
print("\nDtypes:")
print(spending_raw.dtypes)
print("\nHead:")
spending_raw.head()


=== DOT_Spending_Merged.csv ===
Shape: (1178715, 5)

Columns: ['Fiscal year', 'Agency', 'Budget Code', 'Expense Category', 'Check Amount']

Dtypes:
Fiscal year           int64
Agency                  str
Budget Code             str
Expense Category        str
Check Amount        float64
dtype: object

Head:


,Fiscal year,Agency,Budget Code,Expense Category,Check Amount
0,2018,Department of Transportation,HMMN (MANHATTAN BRIDGE HAZARD MITIGATION INCL:),IOTB CONSTRUCTION,86000000.0
1,2026,Department of Transportation,BKHZ (BROOKLYN BRIDGE: HAZARD MITIGATION INCL:),IOTB CONSTRUCTION,85000000.0
2,2026,Department of Transportation,BKHZ (BROOKLYN BRIDGE: HAZARD MITIGATION INCL:),IOTB CONSTRUCTION,85000000.0
3,2026,Department of Transportation,BKHZ (BROOKLYN BRIDGE: HAZARD MITIGATION INCL:),IOTB CONSTRUCTION,85000000.0
4,2026,Department of Transportation,BKHZ (BROOKLYN BRIDGE: HAZARD MITIGATION INCL:),IOTB CONSTRUCTION,85000000.0


## Step 2: Clean

- Convert any comma/currency-formatted string columns to numeric.
- Parse fiscal year as `int`.
- Report missing values.

In [4]:
def clean_currency(series: pd.Series) -> pd.Series:
    """Convert a column to numeric, stripping $ and , if it's stored as text.
    Leaves already-numeric columns untouched."""
    if pd.api.types.is_numeric_dtype(series):
        return series
    cleaned = (
        series.astype(str)
        .str.replace(r"[\$,]", "", regex=True)
        .str.strip()
    )
    return pd.to_numeric(cleaned, errors="coerce")


budget = budget_raw.copy()
spending = spending_raw.copy()

# --- Budget file ---
print("BUDGET currency columns before cleaning:")
print(budget[["Adopted", "Modified", "Post Adjustments", "Pre-Encumbered"]].dtypes)

for col in ["Adopted", "Modified", "Post Adjustments", "Pre-Encumbered"]:
    budget[col] = clean_currency(budget[col])

print("\nBUDGET currency columns after cleaning:")
print(budget[["Adopted", "Modified", "Post Adjustments", "Pre-Encumbered"]].dtypes)

budget["Year"] = pd.to_numeric(budget["Year"], errors="coerce").astype("Int64")

print("\nBUDGET missing values by column:")
print(budget.isna().sum())

# --- Correction 1 (applied): FY2022 multi-agency contamination ---
# Confirmed with evidence in the "Data Quality Investigation" section at the
# end of this notebook (Issue 1): the FY2022 source file for this export was
# not pre-filtered to DOT the way every other fiscal year's file was, and
# pulled in ~150 other agencies alongside real DOT rows. Filtering to
# Agency == "Department of Transportation" is a no-op for every year except
# 2022, where it removes the contamination using rows that were already
# present and correctly labeled -- nothing is dropped or hand-edited.
rows_before = len(budget)
adopted_before = budget["Adopted"].sum()
budget = budget[budget["Agency"] == "Department of Transportation"].copy()

print("\nApplied Agency == 'Department of Transportation' filter:")
print(f"  Rows:                       {rows_before:,} -> {len(budget):,}")
print(f"  Adopted total (all years):  {adopted_before:,.0f} -> {budget['Adopted'].sum():,.0f}")


BUDGET currency columns before cleaning:
Adopted             float64
Modified            float64
Post Adjustments    float64
Pre-Encumbered      float64
dtype: object

BUDGET currency columns after cleaning:
Adopted             float64
Modified            float64
Post Adjustments    float64
Pre-Encumbered      float64
dtype: object

BUDGET missing values by column:
Adopted             0
Agency              0
Budget Code         0
Expense Category    0
Modified            0
Post Adjustments    0
Pre-Encumbered      0
Year                0
Source_File         0
dtype: int64

Applied Agency == 'Department of Transportation' filter:
  Rows:                       334,806 -> 172,576
  Adopted total (all years):  113,208,924,304 -> 13,859,876,192


In [5]:
# --- Spending file ---
print("SPENDING currency column before cleaning:")
print(spending[["Check Amount"]].dtypes)

spending["Check Amount"] = clean_currency(spending["Check Amount"])

print("\nSPENDING currency column after cleaning:")
print(spending[["Check Amount"]].dtypes)

spending["Fiscal year"] = pd.to_numeric(spending["Fiscal year"], errors="coerce").astype("Int64")

print("\nSPENDING missing values by column:")
print(spending.isna().sum())

print(
    "\nNote: 'Budget Code' and 'Expense Category' have missing values in the "
    "spending file, but these are identifier/label columns, not the amount "
    "column being aggregated in Step 3, so they don't block the fiscal-year sums."
)

# --- Correction 2 (applied): expense-vs-capital scope alignment ---
# Confirmed with evidence in the "Data Quality Investigation" section at the
# end of this notebook (Issue 2): DOT_Spending_Merged.csv mixes
# operating-expense and capital spending, while DOT_Budget_2017_2027.csv is
# expense-only. Keep only spending rows whose Budget Code (leading token,
# since the field sometimes carries a parenthetical project name) exists in
# the DOT operating budget file's set of budget codes.
dot_budget_codes = set(budget["Budget Code"].astype(str).str.strip())
spending_code_lead = spending["Budget Code"].astype(str).str.extract(r"^([^\s(]+)")[0]
is_expense = spending_code_lead.isin(dot_budget_codes)

rows_before = len(spending)
amount_before = spending["Check Amount"].sum()
spending = spending[is_expense].copy()

print("\nApplied expense-only Budget Code filter (code must exist in DOT operating budget):")
print(f"  Rows:         {rows_before:,} -> {len(spending):,}")
print(f"  Check Amount: {amount_before:,.0f} -> {spending['Check Amount'].sum():,.0f}")


SPENDING currency column before cleaning:
Check Amount    float64
dtype: object

SPENDING currency column after cleaning:
Check Amount    float64
dtype: object

SPENDING missing values by column:
Fiscal year             0
Agency                  0
Budget Code         19611
Expense Category     7711
Check Amount            0
dtype: int64

Note: 'Budget Code' and 'Expense Category' have missing values in the spending file, but these are identifier/label columns, not the amount column being aggregated in Step 3, so they don't block the fiscal-year sums.



Applied expense-only Budget Code filter (code must exist in DOT operating budget):
  Rows:         1,178,715 -> 1,049,180
  Check Amount: 31,349,477,871 -> 15,153,258,376


## Methodology Note: Corrections Applied to Steps 3–5

Two corrections, both validated with evidence in the *Data Quality
Investigation* section at the end of this notebook, are applied above in
Step 2, before any aggregation:

1. **FY2022 multi-agency contamination.** The FY2022 slice of the budget
   file was not scoped to DOT the way every other fiscal year's file was —
   it included rows from roughly 150 other NYC agencies (Health, Education,
   Police, pension and reserve funds, etc.), inflating FY2022 Adopted from a
   plausible ~$1.3B to an implausible ~$100.6B. **Fix:** filter the budget
   dataframe to `Agency == "Department of Transportation"` immediately
   after cleaning. This is a no-op for FY2017–2021 and FY2023–2027 (already
   DOT-only) and corrects FY2022 using rows that were already present and
   correctly labeled in the same file — no rows were dropped or hand-edited.

2. **Expense-vs-capital scope mismatch.** `DOT_Spending_Merged.csv` mixes
   operating-expense spending (payroll, supplies, contractual services)
   with capital-project spending (`IOTB CONSTRUCTION`, land acquisition,
   building construction), while `DOT_Budget_2017_2027.csv` is an
   expense-only budget. Comparing total spending to modified budget without
   accounting for this made `actual` look like 1.3–2.1x `modified` in every
   year. **Fix:** keep only spending rows whose `Budget Code` (leading
   token, since the field sometimes carries a parenthetical project name)
   exists in the DOT operating budget's set of budget codes.

See *Data Quality Investigation* below for the full evidence trail behind
both corrections.

## Step 3: Aggregate to Fiscal Year

Budget file already covers exactly FY2017–FY2027. The spending file covers
FY2010–FY2027, so rows outside FY2017–FY2027 are dropped here to match the
project scope (reported below).

In [6]:
budget_by_year = (
    budget[budget["Year"].between(2017, 2027)]
    .groupby("Year")[["Adopted", "Modified"]]
    .sum()
    .rename_axis("fiscal_year")
)

print("Adopted + Modified budget by fiscal year:")
print(budget_by_year)


Adopted + Modified budget by fiscal year:
                  Adopted      Modified
fiscal_year                            
2017         9.462619e+08  9.870048e+08
2018         9.680434e+08  1.006587e+09
2019         1.042719e+09  1.067013e+09
2020         1.104236e+09  1.117602e+09
2021         1.099874e+09  1.155962e+09
2022         1.265808e+09  1.294564e+09
2023         1.438489e+09  1.448999e+09
2024         1.405342e+09  1.476735e+09
2025         1.449323e+09  1.568194e+09
2026         1.503042e+09  1.599799e+09
2027         1.636737e+09  1.639747e+09


In [7]:
in_scope = spending["Fiscal year"].between(2017, 2027)
dropped = (~in_scope).sum()
print(f"Spending rows outside FY2017-2027 dropped: {dropped} of {len(spending)}")

spending_by_year = (
    spending[in_scope]
    .groupby("Fiscal year")["Check Amount"]
    .sum()
    .rename_axis("fiscal_year")
    .rename("actual")
)

print("\nActual spending by fiscal year:")
print(spending_by_year)


Spending rows outside FY2017-2027 dropped: 328645 of 1049180

Actual spending by fiscal year:
fiscal_year
2017    9.019302e+08
2018    9.371839e+08
2019    9.530007e+08
2020    9.974097e+08
2021    9.623433e+08
2022    1.105430e+09
2023    1.222150e+09
2024    1.353530e+09
2025    1.423722e+09
2026    1.437636e+09
2027    1.619809e+08
Name: actual, dtype: float64


In [8]:
# --- Unit / scale sanity check: do NOT assume budget and spending are on the
# same scale just because both are numeric. Compare magnitudes directly.
# (Runs on the already-corrected budget/spending from Step 2, so this now
# checks whether the corrections held, not the original raw mismatch.)
comparison = pd.DataFrame({
    "adopted": budget_by_year["Adopted"],
    "modified": budget_by_year["Modified"],
    "actual": spending_by_year,
})
comparison["actual_/_modified"] = comparison["actual"] / comparison["modified"]

print("Budget vs. spending magnitude check (post-correction):")
print(comparison)

flags = []
high_ratio_years = comparison[comparison["actual_/_modified"] > 5]
low_ratio_years = comparison[comparison["actual_/_modified"] < 0.2]

if not high_ratio_years.empty:
    flags.append(
        f"Actual spending is more than 5x modified budget in fiscal year(s) "
        f"{list(high_ratio_years.index)}. Expense/capital scope is already "
        f"aligned (see Methodology Note above), so this would need fresh "
        f"investigation rather than being written off as a scope or units issue."
    )

if not low_ratio_years.empty:
    flags.append(
        f"Actual spending is under 20% of modified budget in fiscal year(s) "
        f"{list(low_ratio_years.index)}. For FY2027 this is expected, not a "
        f"data issue: the fiscal year has barely started as of 2026-08-06, so "
        f"only a few weeks of actual spending exist against a full-year budget."
    )

outlier_years = budget_by_year[budget_by_year["Adopted"] > budget_by_year["Adopted"].median() * 10]
if not outlier_years.empty:
    flags.append(
        f"Adopted budget has an extreme outlier in fiscal year(s) "
        f"{list(outlier_years.index)}: {outlier_years['Adopted'].to_dict()} "
        f"-- roughly 100x the other years. Looks like a data quality issue "
        f"in the source file, not a real budget figure. Recommend inspecting "
        f"the raw rows for that year before trusting downstream metrics."
    )

print("\nFLAGS:")
if flags:
    for f in flags:
        print("- " + f)
else:
    print("None.")


Budget vs. spending magnitude check (post-correction):
                  adopted      modified        actual  actual_/_modified
fiscal_year                                                             
2017         9.462619e+08  9.870048e+08  9.019302e+08           0.913805
2018         9.680434e+08  1.006587e+09  9.371839e+08           0.931051
2019         1.042719e+09  1.067013e+09  9.530007e+08           0.893148
2020         1.104236e+09  1.117602e+09  9.974097e+08           0.892455
2021         1.099874e+09  1.155962e+09  9.623433e+08           0.832504
2022         1.265808e+09  1.294564e+09  1.105430e+09           0.853902
2023         1.438489e+09  1.448999e+09  1.222150e+09           0.843445
2024         1.405342e+09  1.476735e+09  1.353530e+09           0.916570
2025         1.449323e+09  1.568194e+09  1.423722e+09           0.907874
2026         1.503042e+09  1.599799e+09  1.437636e+09           0.898635
2027         1.636737e+09  1.639747e+09  1.619809e+08           0.098

## Step 4: Merge

Join budget and spending on fiscal year.

In [9]:
merged = (
    budget_by_year
    .rename(columns={"Adopted": "adopted", "Modified": "modified"})
    .join(spending_by_year, how="outer")
    .reset_index()
    .rename(columns={"index": "fiscal_year"})
    .sort_values("fiscal_year")
    .reset_index(drop=True)
)

print("Merged shape:", merged.shape)
merged


Merged shape: (11, 4)


,fiscal_year,adopted,modified,actual
0,2017,9.462619e+08,9.870048e+08,9.019302e+08
1,2018,9.680434e+08,1.006587e+09,9.371839e+08
2,2019,1.042719e+09,1.067013e+09,9.530007e+08
3,2020,1.104236e+09,1.117602e+09,9.974097e+08
4,2021,1.099874e+09,1.155962e+09,9.623433e+08
5,2022,1.265808e+09,1.294564e+09,1.105430e+09
6,2023,1.438489e+09,1.448999e+09,1.222150e+09
7,2024,1.405342e+09,1.476735e+09,1.353530e+09
8,2025,1.449323e+09,1.568194e+09,1.423722e+09
9,2026,1.503042e+09,1.599799e+09,1.437636e+09


## Step 5: Compute Metrics

- `modification` = modified - adopted
- `funding_gap` = modified - actual (positive = unspent)
- `spending_efficiency` = actual / modified

In [10]:
final = merged.copy()
final["modification"] = final["modified"] - final["adopted"]
final["funding_gap"] = final["modified"] - final["actual"]
final["spending_efficiency"] = final["actual"] / final["modified"]

pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
print("=== FY2017-2027 DOT Budget vs. Spending: Final Table ===")
final


=== FY2017-2027 DOT Budget vs. Spending: Final Table ===


,fiscal_year,adopted,modified,actual,modification,funding_gap,spending_efficiency
0,2017,"946,261,935.00","987,004,764.00","901,930,214.87","40,742,829.00","85,074,549.13",0.91
1,2018,"968,043,444.00","1,006,587,224.00","937,183,870.64","38,543,780.00","69,403,353.36",0.93
2,2019,"1,042,719,292.00","1,067,012,833.00","953,000,693.10","24,293,541.00","114,012,139.90",0.89
3,2020,"1,104,236,297.00","1,117,601,805.00","997,409,678.73","13,365,508.00","120,192,126.27",0.89
4,2021,"1,099,873,821.00","1,155,962,285.00","962,343,265.13","56,088,464.00","193,619,019.87",0.83
5,2022,"1,265,808,124.00","1,294,564,083.00","1,105,430,478.04","28,755,959.00","189,133,604.96",0.85
6,2023,"1,438,489,469.00","1,448,998,543.00","1,222,150,442.62","10,509,074.00","226,848,100.38",0.84
7,2024,"1,405,341,510.00","1,476,734,619.00","1,353,530,012.62","71,393,109.00","123,204,606.38",0.92
8,2025,"1,449,323,202.00","1,568,194,105.00","1,423,722,022.97","118,870,903.00","144,472,082.03",0.91
9,2026,"1,503,042,033.00","1,599,799,084.00","1,437,635,892.24","96,757,051.00","162,163,191.76",0.90


---
**Stopping here per instructions.** Table above is `adopted`, `modified`,
`actual`, `modification`, `funding_gap`, `spending_efficiency` by fiscal
year, computed on the corrected data (FY2022 restricted to DOT; spending
restricted to expense scope — see the Methodology Note above Step 3, and
the Data Quality Investigation below for the full evidence trail).

---
# Data Quality Investigation (evidence trail)

Two issues were originally flagged in the Step 5 table and are investigated
here, one piece of evidence at a time, using the **raw (unfiltered)** data
so the contamination stays visible regardless of the corrections applied
above. **Both fixes are now applied permanently in Step 2** (see the
Methodology Note before Step 3) — this section is preserved as the
evidence trail that justified them, not a live analysis still pending
action.

## Issue 1: FY2022 outlier

Filter the budget file to fiscal year 2022, sort by `Adopted` descending,
and look at what's actually in the top rows.

In [11]:
b2022 = budget_raw[budget_raw["Year"] == 2022].copy()
print("FY2022 row count (raw, before the Agency filter applied in Step 2):", len(b2022))

top20_adopted = b2022.sort_values("Adopted", ascending=False).head(20)
print("\nTop 20 FY2022 rows by Adopted:")
top20_adopted[["Budget Code", "Expense Category", "Adopted", "Modified", "Agency", "Source_File"]]


FY2022 row count (raw, before the Agency filter applied in Step 2): 177922

Top 20 FY2022 rows by Adopted:


,Budget Code,Expense Category,Adopted,Modified,Agency,Source_File
75226,9564,MEDICAL ASSISTANCE,"5,584,533,142.00","5,972,433,142.00",Department of Social Services,nyc-data-feed DOT 2022.csv
75227,4301,FULL TIME PEDAGOGICAL PRSONNEL,"3,690,469,295.00","3,624,986,972.00",Department of Education,nyc-data-feed DOT 2022.csv
75229,0400,TEACH RET SYS CONTINGNT RES SY,"3,027,809,126.00","3,030,749,653.00",Pension Contributions,nyc-data-feed DOT 2022.csv
75232,0560,POLICE ACTUARIAL PENSION FUND,"2,554,021,068.00","2,462,856,057.00",Pension Contributions,nyc-data-feed DOT 2022.csv
75234,0980,CONTINGENT RESERVE FUND,"2,291,665,852.00","2,267,059,705.00",Pension Contributions,nyc-data-feed DOT 2022.csv
75233,2301,CHARTER SCHOOLS,"2,107,513,522.00","2,267,897,585.00",Department of Education,nyc-data-feed DOT 2022.csv
75236,4601,FULL TIME PEDAGOGICAL PRSONNEL,"2,098,496,934.00","1,826,563,707.00",Department of Education,nyc-data-feed DOT 2022.csv
75237,0990,HEALTH INSURANCE PLAN CITY EMP,"1,906,023,972.00","1,669,552,307.00",Department of Education,nyc-data-feed DOT 2022.csv
75230,3004,HEALTH INSURANCE PLAN CITY EMP,"1,693,883,896.00","2,698,764,895.00",Miscellaneous,nyc-data-feed DOT 2022.csv
75231,3006,HEALTH INSURANCE PLAN CITY EMP,"1,643,370,154.00","2,478,370,154.00",Miscellaneous,nyc-data-feed DOT 2022.csv


In [12]:
top20_modified = b2022.sort_values("Modified", ascending=False).head(20)
print("Top 20 FY2022 rows by Modified (for comparison):")
top20_modified[["Budget Code", "Expense Category", "Adopted", "Modified", "Agency", "Source_File"]]


Top 20 FY2022 rows by Modified (for comparison):


,Budget Code,Expense Category,Adopted,Modified,Agency,Source_File
75226,9564,MEDICAL ASSISTANCE,"5,584,533,142.00","5,972,433,142.00",Department of Social Services,nyc-data-feed DOT 2022.csv
75227,4301,FULL TIME PEDAGOGICAL PRSONNEL,"3,690,469,295.00","3,624,986,972.00",Department of Education,nyc-data-feed DOT 2022.csv
75228,4001,INTEREST ON BONDS - GENERAL,0.00,"3,317,689,787.00",Debt Service,nyc-data-feed DOT 2022.csv
75229,0400,TEACH RET SYS CONTINGNT RES SY,"3,027,809,126.00","3,030,749,653.00",Pension Contributions,nyc-data-feed DOT 2022.csv
75230,3004,HEALTH INSURANCE PLAN CITY EMP,"1,693,883,896.00","2,698,764,895.00",Miscellaneous,nyc-data-feed DOT 2022.csv
75231,3006,HEALTH INSURANCE PLAN CITY EMP,"1,643,370,154.00","2,478,370,154.00",Miscellaneous,nyc-data-feed DOT 2022.csv
75232,0560,POLICE ACTUARIAL PENSION FUND,"2,554,021,068.00","2,462,856,057.00",Pension Contributions,nyc-data-feed DOT 2022.csv
75233,2301,CHARTER SCHOOLS,"2,107,513,522.00","2,267,897,585.00",Department of Education,nyc-data-feed DOT 2022.csv
75234,0980,CONTINGENT RESERVE FUND,"2,291,665,852.00","2,267,059,705.00",Pension Contributions,nyc-data-feed DOT 2022.csv
75235,6001,INTEREST ON BONDS - GENERAL,0.00,"1,964,685,692.00",Debt Service,nyc-data-feed DOT 2022.csv


In [13]:
# The top rows are NOT DOT expense categories (MEDICAL ASSISTANCE, FULL TIME
# PEDAGOGICAL PERSONNEL, POLICE ACTUARIAL PENSION FUND, ...). That's a
# strong signal this isn't "a few bad DOT rows" -- check the Agency column.

print("Agency breakdown for ALL FY2022 rows (raw, not just top 20):")
print(b2022["Agency"].value_counts())

print("\nSource_File values within FY2022 (is it one bad ingest?):")
print(b2022["Source_File"].value_counts())


Agency breakdown for ALL FY2022 rows (raw, not just top 20):
Agency
Department of Health and Mental Hygiene    19985
Department of Transportation               15692
Department of Education                    12922
Department of Parks and Recreation         11423
Police Department                          11159
                                           ...  
LEASE ADJUSTMENT                               2
GENERAL RESERVE                                2
PRIOR YEAR PAYABLES                            2
FRINGE BENEFITS  COST CONTAINMENT              1
FEDERAL / STATE ACTIONS                        1
Name: count, Length: 151, dtype: int64

Source_File values within FY2022 (is it one bad ingest?):
Source_File
nyc-data-feed DOT 2022.csv    177922
Name: count, dtype: int64


In [14]:
# Is this isolated to FY2022, or does every year's source file mix agencies?
by_year_total = budget_raw.groupby("Year").size()
by_year_dot = budget_raw[budget_raw["Agency"] == "Department of Transportation"].groupby("Year").size()

agency_scope_check = pd.DataFrame({
    "total_rows": by_year_total,
    "dot_rows": by_year_dot,
})
agency_scope_check["non_dot_rows"] = agency_scope_check["total_rows"] - agency_scope_check["dot_rows"]
print("Row counts, all agencies vs. DOT-only, by fiscal year (raw data):")
agency_scope_check


Row counts, all agencies vs. DOT-only, by fiscal year (raw data):


,total_rows,dot_rows,non_dot_rows
Year,,,
2017,14626,14626,0
2018,14851,14851,0
2019,15072,15072,0
2020,15257,15257,0
2021,15420,15420,0
2022,177922,15692,162230
2023,15956,15956,0
2024,16128,16128,0
2025,16356,16356,0


In [15]:
# Quantify the effect on Adopted totals: all-agency sum vs DOT-only sum, per year
all_agency_adopted = budget_raw.groupby("Year")["Adopted"].sum()
dot_only_adopted = (
    budget_raw[budget_raw["Agency"] == "Department of Transportation"]
    .groupby("Year")["Adopted"]
    .sum()
)
issue1_evidence = pd.DataFrame({
    "all_agency_adopted": all_agency_adopted,
    "dot_only_adopted": dot_only_adopted,
})
print("Adopted budget total: all agencies included vs. DOT filtered, by year (raw data):")
issue1_evidence


Adopted budget total: all agencies included vs. DOT filtered, by year (raw data):


,all_agency_adopted,dot_only_adopted
Year,,
2017,"946,261,935.00","946,261,935.00"
2018,"968,043,444.00","968,043,444.00"
2019,"1,042,719,292.00","1,042,719,292.00"
2020,"1,104,236,297.00","1,104,236,297.00"
2021,"1,099,873,821.00","1,099,873,821.00"
2022,"100,614,856,236.00","1,265,808,124.00"
2023,"1,438,489,469.00","1,438,489,469.00"
2024,"1,405,341,510.00","1,405,341,510.00"
2025,"1,449,323,202.00","1,449,323,202.00"


**Finding — Issue 1:**

FY2022's source file (`nyc-data-feed DOT 2022.csv`) is the *only* year that
was not pre-filtered to DOT — it contains **162,230 rows from 150 other
agencies** (Health, Education, Police, Fire, pensions, general reserves,
etc.) alongside the 15,692 real DOT rows. Every other fiscal year's file
(2017–2021, 2023–2027) contains DOT rows exclusively — `non_dot_rows` is 0
for all of them.

This is **not** a few bad rows and **not** a units problem — it's a
systematic scope error in one year's source file: the full citywide budget
got merged into the FY2022 DOT export. Filtering FY2022 to
`Agency == "Department of Transportation"` brings Adopted from **$100.6B
down to $1.27B**, which fits cleanly on the trend line between FY2021
($1.10B) and FY2023 ($1.44B).

**Fix:** don't drop FY2022 and don't hand-edit a row. Apply
`Agency == "Department of Transportation"` as a filter to the *entire*
budget dataframe before aggregating. It's a no-op for every other year
(already DOT-only) and fixes FY2022 using data that's already correctly
labeled in the same file.

**Applied.** This filter is now in place in Step 2 above, before Step 3's
aggregation.

## Issue 2: scope mismatch (actual ≈ 1.3–2.1x modified)

List the unique category/budget-code labels in the spending file and look
for what separates capital-type spending (construction, bridges, IOTB) from
expense-type spending (payroll, supplies, contractual services).

In [16]:
print("Unique 'Expense Category' values in DOT_Spending_Merged.csv (raw, pre-filter):", spending_raw["Expense Category"].nunique(dropna=True))
print()
print("All values, by row count:")
spending_raw["Expense Category"].value_counts(dropna=False)


Unique 'Expense Category' values in DOT_Spending_Merged.csv (raw, pre-filter): 86

All values, by row count:


Expense Category
Payroll Summary                   436215
SUPPLIES + MATERIALS - GENERAL    136555
RENTALS OF MISC.EQUIP             110617
AUTOMOTIVE SUPPLIES & MATERIAL     94401
IOTB CONSTRUCTION                  72177
                                   ...  
RELOCATION EXPENSES                    2
MEDICAL,SURGICAL & LAB SUPPLY          2
OTHER EXPENDITURES-GENERAL             1
IOTB SITE ACQUISITION                  1
MEDICAL ASSISTANCE                     1
Name: count, Length: 87, dtype: int64

In [17]:
print("Unique 'Budget Code' values in DOT_Spending_Merged.csv (raw, pre-filter):", spending_raw["Budget Code"].nunique(dropna=True))
print()
print("Sample of Budget Code values (sorted):")
sample_codes = sorted(spending_raw["Budget Code"].dropna().unique())
print("First 15:", sample_codes[:15])
print("...")
print("A slice further in (shows the parenthetical-project-name pattern):")
print([c for c in sample_codes if "(" in c][:15])


Unique 'Budget Code' values in DOT_Spending_Merged.csv (raw, pre-filter): 2645

Sample of Budget Code values (sorted):


First 15: ['001J', '001M', '002M', '006J', '029A (DOT: 424 WYTHE AVE, BK: CON NEW PAINT WA)', '029B (DOT: 424 WYTHE AVE, BK: PARKING LOT (ASP)', '048D (BROOKLYN WATERFRONT GREENWAY, SUNSET PAR)', '048E (BROOKLYN WATERFRONT GREENWAY (FROM COMME)', '0KMC (VAR LOC, CITY-WIDE: CONST COMPLEX PED RA)', '0MQC (VAR LOC, CITY-WIDE: CONST COMPLEX PED RA)', '1000', '1000 (<UNKNOWN>)', '1000 (ADMINISTRATION)', '1000 (ADMINISTRATIVE PS)', '1000 (OFF OF THE COMMISSIONER)']
...
A slice further in (shows the parenthetical-project-name pattern):
['029A (DOT: 424 WYTHE AVE, BK: CON NEW PAINT WA)', '029B (DOT: 424 WYTHE AVE, BK: PARKING LOT (ASP)', '048D (BROOKLYN WATERFRONT GREENWAY, SUNSET PAR)', '048E (BROOKLYN WATERFRONT GREENWAY (FROM COMME)', '0KMC (VAR LOC, CITY-WIDE: CONST COMPLEX PED RA)', '0MQC (VAR LOC, CITY-WIDE: CONST COMPLEX PED RA)', '1000 (<UNKNOWN>)', '1000 (ADMINISTRATION)', '1000 (ADMINISTRATIVE PS)', '1000 (OFF OF THE COMMISSIONER)', '1000 (TRACK)', '1007 (SPECIAL EVENTS, CITY)', '100

In [18]:
# The Budget Code field mixes two formats: a bare code ("4125") and a code
# with a parenthetical note ("048D (BROOKLYN WATERFRONT GREENWAY...)").
# Extract the leading token (before the first space/paren) so both formats
# compare on the same basis, then check whether that leading code appears
# anywhere in the DOT *operating* budget file at all.
#
# Uses the raw (pre-filter) spending and budget files so this evidence
# stands on its own, independent of the corrections already applied in
# Step 2.

spending_investigate = spending_raw.copy()
lead_code = spending_investigate["Budget Code"].astype(str).str.extract(r"^([^\s(]+)")[0]
spending_investigate["code_lead"] = lead_code

dot_budget_codes_raw = set(
    budget_raw[budget_raw["Agency"] == "Department of Transportation"]["Budget Code"]
    .astype(str)
    .str.strip()
)
spending_investigate["code_in_operating_budget"] = spending_investigate["code_lead"].isin(dot_budget_codes_raw)

print("Distinct Budget Codes in the DOT operating budget file:", len(dot_budget_codes_raw))
print()
print("Spending rows: does the code exist in the operating budget file?")
print(spending_investigate["code_in_operating_budget"].value_counts())
print()
print("Check Amount total, split by whether the code is in the operating budget file:")
print(spending_investigate.groupby("code_in_operating_budget")["Check Amount"].sum())


Distinct Budget Codes in the DOT operating budget file: 995

Spending rows: does the code exist in the operating budget file?
code_in_operating_budget
True     1049180
False     129535
Name: count, dtype: int64

Check Amount total, split by whether the code is in the operating budget file:
code_in_operating_budget
False   16,196,219,495.26
True    15,153,258,375.85
Name: Check Amount, dtype: float64


In [19]:
print("Top Expense Category values where the code is NOT in the operating budget file")
print("(codes with no operating-budget match -- candidate CAPITAL spending):")
spending_investigate[~spending_investigate["code_in_operating_budget"]]["Expense Category"].value_counts().head(15)


Top Expense Category values where the code is NOT in the operating budget file
(codes with no operating-budget match -- candidate CAPITAL spending):


Expense Category
IOTB CONSTRUCTION                      71250
DESIGN-CONSULTANT-IOTB                 35061
CAPITAL PURCHASED EQUIPMENT             6008
<Non-Applicable Expenditure Object>     2549
LAND ACQUISITION - CONDEMNATION         1864
CONSTRUCTION-BUILDINGS                  1150
INTEREST ON LAND ACQUISITION             993
INCIDENTAL COSTS                         619
Payroll Summary                          525
DESIGN-CONSULTANT-BUILDINGS              366
PROF SERV ENGINEER & ARCHITECT           337
POLLUTION REMEDIATION OBLIGATIONS        304
DESIGN-BUILD: INFRASTRUCTURE             188
DEMOLITION                                77
SUPPLIES + MATERIALS - GENERAL            72
Name: count, dtype: int64

In [20]:
print("Top Expense Category values where the code IS in the operating budget file")
print("(codes that match the expense budget -- candidate EXPENSE spending):")
spending_investigate[spending_investigate["code_in_operating_budget"]]["Expense Category"].value_counts().head(15)


Top Expense Category values where the code IS in the operating budget file
(codes that match the expense budget -- candidate EXPENSE spending):


Expense Category
Payroll Summary                   435690
SUPPLIES + MATERIALS - GENERAL    136483
RENTALS OF MISC.EQUIP             110556
AUTOMOTIVE SUPPLIES & MATERIAL     94357
MAINTENANCE SUPPLIES               64226
MAINT & REP GENERAL                35920
MAINT & OPER OF INFRASTRUCTURE     27238
CONTRACTUAL SERVICES GENERAL       23153
PROF SERV OTHER                    12625
MOTOR VEHICLE FUEL                 11809
OFF SVC-MEMBERSHIP DUES & FEES     10723
RENTALS - LAND BLDGS & STRUCTS      9086
EQUIPMENT GENERAL                   7132
PROMPT PAYMENT INTEREST             7034
CLEANING SERVICES                   5512
Name: count, dtype: int64

**Finding — Issue 2:**

The split is close to clean. Spending rows whose `Budget Code` matches a
code in the DOT operating (expense) budget file are dominated by
`Payroll Summary`, `SUPPLIES + MATERIALS`, `RENTALS`, `MAINTENANCE`,
`CONTRACTUAL SERVICES` — textbook operating-expense categories. Rows whose
code does *not* appear in the operating budget file are dominated by
`IOTB CONSTRUCTION`, `DESIGN-CONSULTANT-IOTB`, `CAPITAL PURCHASED
EQUIPMENT`, `LAND ACQUISITION - CONDEMNATION`, `CONSTRUCTION-BUILDINGS` —
textbook capital-project categories. `DOT_Budget_2017_2027.csv` itself
contains no capital-style categories at all (confirmed above: it's
expense-only, no `IOTB`/`CONSTRUCTION`/`CAPITAL` labels in the whole file).

So: yes, `DOT_Spending_Merged.csv` mixes expense and capital spending,
while the budget file is expense-only — confirmed scope mismatch, not a
units mismatch.

**Applied.** This filter is now in place in Step 2 above, before Step 3's
aggregation.

In [21]:
# Replication check: recompute the corrected year-by-year table directly
# from the raw data plus both corrections, independent of the Step 2-5
# pipeline above, and confirm it matches. This is a check, not a second
# source of truth.

budget_dot_only_check = budget_raw[budget_raw["Agency"] == "Department of Transportation"]
budget_by_year_check = (
    budget_dot_only_check[budget_dot_only_check["Year"].between(2017, 2027)]
    .groupby("Year")[["Adopted", "Modified"]]
    .sum()
)

spending_expense_only_check = spending_investigate[
    spending_investigate["Fiscal year"].between(2017, 2027)
    & spending_investigate["code_in_operating_budget"]
]
spending_by_year_check = spending_expense_only_check.groupby("Fiscal year")["Check Amount"].sum()

check_table = pd.DataFrame({
    "adopted": budget_by_year_check["Adopted"],
    "modified": budget_by_year_check["Modified"],
    "actual_expense_only": spending_by_year_check,
})
check_table["spending_efficiency_check"] = (
    check_table["actual_expense_only"] / check_table["modified"]
)
print("Independent replication of the corrected table (should match Step 5 above):")
check_table


Independent replication of the corrected table (should match Step 5 above):


,adopted,modified,actual_expense_only,spending_efficiency_check
2017,"946,261,935.00","987,004,764.00","901,930,214.87",0.91
2018,"968,043,444.00","1,006,587,224.00","937,183,870.64",0.93
2019,"1,042,719,292.00","1,067,012,833.00","953,000,693.10",0.89
2020,"1,104,236,297.00","1,117,601,805.00","997,409,678.73",0.89
2021,"1,099,873,821.00","1,155,962,285.00","962,343,265.13",0.83
2022,"1,265,808,124.00","1,294,564,083.00","1,105,430,478.04",0.85
2023,"1,438,489,469.00","1,448,998,543.00","1,222,150,442.62",0.84
2024,"1,405,341,510.00","1,476,734,619.00","1,353,530,012.62",0.92
2025,"1,449,323,202.00","1,568,194,105.00","1,423,722,022.97",0.91
2026,"1,503,042,033.00","1,599,799,084.00","1,437,635,892.24",0.90


**Confirmed.** This independent replication matches the corrected Step 5
table above (`spending_efficiency` in the ~0.83–0.93 range for
FY2017–2026; FY2027 is lower because the fiscal year has barely started as
of 2026-08-06, not a data issue). Both corrections are applied permanently
in Step 2 — see the Methodology Note before Step 3.